# LS-NEQ 06_geometry_trajectory — Lokale Geometrie und Trajektorie

**Dossier-Stufe G** (`lsne.experiments.STAGES["G"]`, siehe `docs/DESIGN.md` §8 und `docs/STAGES.md`).
Dieses Notebook wird von `scripts/build_notebooks.py` erzeugt; Änderungen bitte dort
vornehmen und neu generieren (`python scripts/build_notebooks.py`).

**Zweck.** Der Zwischenschritt aus `docs/DESIGN.md` §4.4/§4.5 (Godoy Ortiz 2026, *Meaning Contracts*): Realisierungswolken pro Bedingung (Familie, K, C) aus allen Formulierungen × zulässigen Einheitsmasken, lokale Geometrie an jedem Residual-Snapshot (Top-k-Deflation, trunkierte Mahalanobis-Distanz, Log-Volumen, Participation Ratio, Verschiebung zur K=0-Wolke), G0-Gate (Kern-und-Streuung-Ordnung exakter Übersetzungen), G1-Kontraktionskontrast (n-angepasst, Familien-Bootstrap) und die Trajektorien T1 (Tiefe: „atmet“ der Raum?), T2 (Position relativ zur Wechselstelle), T3 (Steering-Validität: reproduziert `lang_dom` die diskrete K-Verschiebung?) und T5 (Kopplung von Δ log-Vol mit λ_ℓ und Lens-KL aus Stufe D).

**Eingaben.** Stimulusbank (alle Formulierungen A–D einer Familie), Richtungsbank (Stufe B auf Drive oder neu); für T5 der Stufe-D-Lauf desselben Modells unter `RUNS_ROOT` (Notebook 03).

**Ausgaben.** Tabellen `clouds`, `geometry`, `pair_ordering`, `g0_gate`, `depth_trajectory`, `contraction`, `contraction_boot`, `position_traj`, `steering`, `steering_boot`, `coupling`.
Alle Tabellen liegen als CSV (+ Parquet) im Laufverzeichnis
`<RUNS_ROOT>/G/<modell>@<revision>/` zusammen mit `manifest.json`
(Git-SHA, Modell, dtype, Hook, Anker-Hashes, Zeitstempel); Abbildungen unter
`<OUT_ROOT>/figures/06_geometry_trajectory/`; am Ende eine kurze `summary_06_geometry_trajectory_*.json` in `OUT_ROOT`.

**Ablauf.** Setup → Drive → Konfiguration → Modell → Stufe G → G0-Gate → T1 Tiefenkurven → G1 Kontraktion mit Bootstrap → T2 Positionsinkremente → T3 Steering-Kosinus → T5 Kopplung → Zusammenfassung.

**Präregistrierung.** Kontrast 7 (G0-Gate: `g0_gate.pass_all` / `pass_middle`; `degenerate = True` heißt: zu wenige Zustände für den Rang, Gate nicht aussagekräftig), Kontrast 8 (G1: `contraction_boot`-Zeilen mit `preregistered = True`, K=8/C1, Deflation k=5, Mean-Pool für Log-Vol, Last-Token für PR, Vorzeichen offen) und Kontrast 9 (T3: `steering_boot`, `quantity = cos_at_max_disp_layer`, ε = 0,05, Mindesteffekt cos ≥ 0,3 gegen 32 Zufallsrichtungen). Fällt G0 durch, werden 8–9 nicht interpretiert; fällt T3 negativ aus, gelten Dosis-Phasendiagramme nur als Eingriffs-, nicht als Sprachwechselbefunde.

Hinweise: Die Zellen sind gegen Fehler abgesichert (ein fehlgeschlagener Schritt druckt
den Traceback und das Notebook läuft weiter). Mit `SMOKE = True` läuft alles in
< 3 min auf CPU mit `pythia-14m` (Rauchtest, keine wissenschaftliche Aussage).
Mit `REUSE = True` werden vorhandene Ergebnisse (manifest.json mit `status: ok`)
geladen statt neu gerechnet — so lässt sich nach einem Colab-Abbruch fortsetzen.


In [ ]:
# --- Setup: Repository holen / aktualisieren und installieren -------------------------
# Runs inside Colab (clone into /content/lsne-repo) and inside an existing checkout
# (local Jupyter, tests): a checkout is detected by lsne/__init__.py + pyproject.toml.
import importlib.util
import os
import pathlib
import subprocess
import sys

GIT_URL = os.environ.get("LSNE_GIT_URL", "https://github.com/Erikiss/Language-Switching-.git")
GIT_BRANCH = os.environ.get("LSNE_GIT_BRANCH", "claude/language-switching-non-equilibrium-ejpz0q")
REPO_DIR = os.environ.get("LSNE_REPO_DIR", "/content/lsne-repo")


def _sh(cmd: str) -> int:
    print("$", cmd, flush=True)
    return subprocess.run(cmd, shell=True).returncode


_here = pathlib.Path.cwd().resolve()
_local = next((p for p in [_here, *_here.parents]
               if (p / "lsne" / "__init__.py").exists() and (p / "pyproject.toml").exists()), None)
if _local is not None:
    REPO_DIR = str(_local)
    print("Vorhandener Checkout:", REPO_DIR)
elif pathlib.Path(REPO_DIR, ".git").exists():
    # a reused runtime: move the shallow checkout to the branch tip (a fast-forward pull fails on
    # depth-1 history and would silently keep the old code)
    if _sh(f"git -C {REPO_DIR} fetch -q --depth 1 origin {GIT_BRANCH} && "
           f"git -C {REPO_DIR} checkout -q -f -B {GIT_BRANCH} FETCH_HEAD") != 0:
        raise RuntimeError(f"{REPO_DIR} konnte nicht auf den Stand von {GIT_BRANCH} gebracht werden: "
                           "Laufzeit -> Sitzung trennen und löschen, dann erneut ausführen")
elif _sh(f"git clone -q --depth 1 --branch {GIT_BRANCH} {GIT_URL} {REPO_DIR}") != 0:
    raise RuntimeError(f"git clone von {GIT_URL} ({GIT_BRANCH}) fehlgeschlagen")
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

if _local is None or importlib.util.find_spec("lsne") is None:
    _sh(f"{sys.executable} -m pip install -q -r requirements.txt")
    _sh(f"{sys.executable} -m pip install -q -e .")
# modules imported by an earlier run in this runtime would shadow the updated checkout
for _m in [m for m in sys.modules if m == "lsne" or m.startswith("lsne.")]:
    del sys.modules[_m]
import lsne
print("lsne", lsne.__version__, "aus", REPO_DIR, "| Branch", GIT_BRANCH)
print("Git-SHA:", subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip())


In [ ]:
# --- Optional: TNCO (Google Research) ---------------------------------------------------
# TNCO optimises the contraction ORDER of the linearised lattice network (stage E).
# It needs the Boost headers and a C++ build (a few minutes on Colab).  Without TNCO
# lsne.tn_lattice.optimize_path falls back to opt_einsum ("dp" for <= 12 tensors,
# else "greedy"); the contracted response is identical, only the path cost / runtime
# columns of tn_paths differ ("fallback_reason" says why).  Stages A-D never need it.
INSTALL_TNCO = False   # True: bauen (nur für Notebook 04 / Stufe E sinnvoll)

try:
    import tnco
    print("tnco", getattr(tnco, "__version__", "?"), "ist bereits installiert")
except ImportError:
    if INSTALL_TNCO:
        _sh("apt-get install -y -q libboost-dev > /tmp/apt_boost.log 2>&1 || cat /tmp/apt_boost.log | tail -5")
        _sh(f'{sys.executable} -m pip install -q "tnco @ git+https://github.com/google-research/tnco"')
        try:
            import tnco
            print("tnco", getattr(tnco, "__version__", "?"), "installiert")
        except ImportError as e:
            print("TNCO konnte nicht gebaut werden:", repr(e), "-> Fallback opt_einsum (siehe docs/RUNBOOK_COLAB.md)")
    else:
        print("TNCO nicht installiert -> lsne.tn_lattice nutzt opt_einsum (dp/greedy) als Fallback.")


In [ ]:
# --- Google Drive einbinden (außerhalb von Colab wird lokal geschrieben) ---------------
# On Colab the results MUST go to Drive: a refused or failed mount stops the notebook
# here (the local disk of the runtime is deleted with the session).  Outside Colab, or
# with LSNE_ALLOW_LOCAL=1, the outputs go to $LSNE_RUNS or ./runs.
import os
import pathlib

try:
    from google.colab import drive            # type: ignore
except ImportError:                           # not on Colab
    drive = None
    print("Kein Colab -> lokale Ausgabe")
if drive is not None and os.environ.get("LSNE_ALLOW_LOCAL") != "1":
    drive.mount("/content/drive")             # confirm the permission dialog in this tab
    if not pathlib.Path("/content/drive/MyDrive").is_dir():
        raise RuntimeError("Google Drive ist nicht eingebunden: die Ergebnisse würden nur auf der "
                           "Laufzeit liegen und mit ihr gelöscht. Diese Zelle erneut ausführen und den "
                           "Drive-Zugriff im Dialog erlauben.")

from lsne.io import drive_root

OUT_ROOT = drive_root()                       # Colab: /content/drive/MyDrive/LSNE, sonst $LSNE_RUNS oder ./runs
RUNS_ROOT = os.path.join(OUT_ROOT, "runs")    # Laufverzeichnisse wie in jobs/shards.yaml (--out)
os.makedirs(RUNS_ROOT, exist_ok=True)
if OUT_ROOT.startswith("/content/drive"):
    # share the anchor / unigram caches between sessions (see jobs/shards.yaml defaults.env)
    os.environ.setdefault("LSNE_CACHE", os.path.join(OUT_ROOT, "cache"))
print("OUT_ROOT  =", OUT_ROOT)
print("RUNS_ROOT =", RUNS_ROOT)
print("LSNE_CACHE =", os.environ.get("LSNE_CACHE", "./cache"))


In [ ]:
# --- Konfiguration (hier anpassen) ------------------------------------------------------
NOTEBOOK = "06_geometry_trajectory"
STAGE = "G"
MODEL = "EleutherAI/pythia-160m"   # HF-Modellname (State 2143: pythia-160m)
REVISION = "step98000"             # HF-Revision: step98000 / step99000 (Eingriff), step143000 (SAE)
DTYPE = "float32"                  # "float32" (Standard) | "bfloat16" (eigener Lauf, eigene Null-Dosis-Baseline)
DEVICE = "auto"                    # "auto" | "cuda" | "cpu"
FAMILIES = None                    # z. B. ["F01", "F02"] oder ["DEV01A"]; None = alle Familien (F01-F12, DEV01-DEV04)
SMOKE = False                      # True: Mini-Lauf mit pythia-14m, 1 Familie, 3 Varianten (< 3 min CPU)
SEED = 0
REUSE = True                       # vorhandene Ergebnisse (manifest.json, status ok) laden statt neu rechnen
USE_SAE = False                    # SAE-Feature-Richtungen (EleutherAI/sae-pythia-160m-32k; nur bei step143000 zertifizierbar)

from lsne.config import ModelSpec, RunConfig
from lsne.experiments import common as C

if SMOKE:
    MODEL, REVISION, DTYPE = "EleutherAI/pythia-14m", "step98000", "float32"
    FAMILIES = FAMILIES or ["DEV01"]
spec = ModelSpec(name=MODEL, revision=REVISION, dtype=DTYPE, device=DEVICE)
cfg = RunConfig(model=spec, families=FAMILIES, seed=SEED, out_root=RUNS_ROOT)
limits = C.Limits()
if SMOKE:
    cfg, limits = C.smoke_config(cfg), C.Limits.smoke()
limits.use_sae = bool(USE_SAE) and not SMOKE
limits.progress = True

_fam = f" --families {','.join(FAMILIES)}" if FAMILIES else ""
CLI_COMMAND = (f"python -m lsne.cli run --stage {STAGE} --model {MODEL} --revision {REVISION} --dtype {DTYPE}"
               f" --device {DEVICE} --out {RUNS_ROOT} --seed {SEED}{_fam}{' --smoke' if SMOKE else ''}"
               f"{' --use-sae' if limits.use_sae else ''}")
print("Modell:", spec.tag(), "| Familien:", FAMILIES or "alle", "| SMOKE:", SMOKE)
print("CLI-Äquivalent:\n  " + CLI_COMMAND)


In [ ]:
# --- Modell laden -------------------------------------------------------------------
import torch
from lsne.model import load_model

mb = load_model(spec)
print(f"{spec.tag()}: {mb.n_layers} Blöcke, d_model={mb.d_model}, {mb.n_heads} Heads, "
      f"Vokabular={mb.vocab}, device={mb.device}, dtype={mb.dtype}")
print("Dossier-Eingriffsstelle (verallgemeinert):", C.dossier_site(mb).key,
      "| Tiefen-Injektion (D1b / Stufe E):", C.depth_site(mb).key)


In [ ]:
# --- Runner: Stufen direkt über run() ausführen (Ergebnisse bleiben im Speicher) --------
import contextlib
import json
import time
import traceback

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

from lsne.experiments import STAGES, STAGE_TITLES
from lsne.io import RunDir, load_df, read_manifest

ctx: dict = {}        # shared objects between stages (variants, original, direction bank)
results: dict = {}    # stage -> {table name: DataFrame}
report: dict = {}     # stage -> status / runtime / error
FIG_DIR = os.path.join(OUT_ROOT, "figures", NOTEBOOK)
os.makedirs(FIG_DIR, exist_ok=True)


def stage_status(stage: str, rd, status: str) -> dict:
    """Report entry of a finished stage: a stale error.txt of an earlier failed run is removed
    (the manifest says ok) and failed substages (G-L, manifest.json -> substages) are listed;
    the status then reads ok_with_substage_errors / reused_with_substage_errors."""
    if C.clear_stale_error(rd.path):
        print(f"Stufe {stage}: veraltete error.txt eines früheren Laufs entfernt ({rd.path})")
    sub = C.substage_status(rd.path, status)
    for name, err in sub["substage_errors"].items():
        print(f"Stufe {stage}: Substufe {name} FEHLER: {err}")
    return sub


def run_stage(stage: str, reuse=None, **kw):
    """Run one stage; on failure print the traceback, write error.txt and continue."""
    reuse = REUSE if reuse is None else reuse
    rd = RunDir(cfg.out_root, stage, cfg.model)
    t0 = time.perf_counter()
    if reuse and rd.manifest_path.exists():
        try:
            m = read_manifest(rd)
            if m.get("status") == "ok" and m.get("tables"):
                results[stage] = {name: load_df(rd, name) for name in m["tables"]}
                report[stage] = dict(run_dir=str(rd.path), seconds=0.0, error="", **stage_status(stage, rd, "reused"))
                print(f"Stufe {stage}: vorhandene Ergebnisse geladen aus {rd.path} "
                      f"(REUSE=False erzwingt Neuberechnung)")
                return results[stage]
        except Exception as e:  # unreadable previous run: recompute
            print("Wiederverwendung nicht möglich:", repr(e))
    print(f"Stufe {stage} ({STAGE_TITLES[stage]}) -> {rd.path}", flush=True)
    try:
        results[stage] = STAGES[stage].run(cfg, mb, rd, ctx=ctx, limits=limits, **kw)
        report[stage] = dict(run_dir=str(rd.path), seconds=time.perf_counter() - t0, error="",
                             **stage_status(stage, rd, "ok"))
        print(f"Stufe {stage} {report[stage]['status']} in {report[stage]['seconds']:.0f} s: "
              f"{', '.join(sorted(results[stage]))}")
    except Exception as e:
        traceback.print_exc()
        C.write_error(rd, stage, e, cfg, mb)
        report[stage] = dict(status="error", run_dir=str(rd.path), seconds=time.perf_counter() - t0, error=repr(e))
        results[stage] = {}
        print(f"Stufe {stage} FEHLGESCHLAGEN: {e!r} (error.txt in {rd.path})")
    return results[stage]


def table(stage: str, name: str) -> pd.DataFrame:
    """Table of a stage result or an empty DataFrame (keeps the plot cells robust)."""
    df = results.get(stage, {}).get(name)
    return df if isinstance(df, pd.DataFrame) else pd.DataFrame()


def show(df, n: int = 40):
    """display() in Jupyter / Colab, print() elsewhere."""
    if isinstance(df, pd.DataFrame):
        df = df.head(n)
    try:
        display(df)          # noqa: F821 - provided by IPython
    except NameError:
        print(df.to_string() if hasattr(df, "to_string") else df)


def savefig(name: str):
    path = os.path.join(FIG_DIR, f"{name}_{spec.tag().replace(':', '_').replace('@', '_')}.png")
    plt.gcf().savefig(path, dpi=120, bbox_inches="tight")
    print("Abbildung:", path)


@contextlib.contextmanager
def guarded(title: str):
    """Print the traceback of a failing cell section and carry on."""
    print("==", title)
    try:
        yield
    except Exception:
        traceback.print_exc()
        print(f"[{title}] übersprungen (siehe Traceback)")


In [ ]:
# --- Substufen-Bericht: die Stufen G-L fangen Fehler pro Substufe ab (docs/STAGES.md) ----
# A failed substage is recorded in manifest.json -> substages.errors (traceback in
# substage_errors.txt), its table stays empty and the stage goes on; the manifest's
# `prereg` map names the table / rows that carry every pre-registered contrast.
def substage_report(stage: str) -> dict:
    """Print the substage report and the pre-registration map of a G-L manifest."""
    rd = RunDir(cfg.out_root, stage, cfg.model)
    if not rd.manifest_path.exists():
        print(f"Stufe {stage}: kein manifest.json in {rd.path}")
        return {}
    m = read_manifest(rd)
    sub = m.get("substages") or {}
    rt = m.get("runtime_s")
    print(f"Stufe {stage}: Status {m.get('status')}" + (f", {rt:.0f} s" if isinstance(rt, (int, float)) else "")
          + f", Substufen fehlgeschlagen: {sub.get('n_failed', '?')}"
          + (" | STUB: lsne.transplant nicht importierbar" if m.get("stub") else ""))
    for name, err in (sub.get("errors") or {}).items():
        print(f"  FEHLER in Substufe {name}: {err}")
    if sub.get("n_failed"):
        print(f"  Tracebacks: {rd.path / 'substage_errors.txt'}")
    for k, v in (m.get("prereg") or {}).items():
        print(f"  Präregistrierung {k}: {v}")
    return m


def truthy(s: pd.Series) -> pd.Series:
    """Boolean view of a flag column (bool in memory, 'True'/'False' strings after a csv reload)."""
    return s.astype(str).str.lower().eq("true")


In [ ]:
# --- Stufe G ausführen ----------------------------------------------------------------
# CLI-Äquivalent:
#   python -m lsne.cli run --stage G --model EleutherAI/pythia-160m --revision step98000 --out <RUNS_ROOT>
# T5 (coupling) reads damage_profile / lens of the stage-D run of the same model below
# RUNS_ROOT (empty until notebook 03 has run); the clouds are shared with H/K via ctx.
print(f"Wolken: K {limits.g_k_levels} × C {limits.g_c_levels}, bis {limits.g_max_masks} Masken je Bedingung, "
      f"Deflation k {limits.g_deflate_k}, Rang r_mean={limits.g_r_mean} / r_last={limits.g_r_last}, "
      f"T3-Dosen {limits.g_steer_eps}, n-Matching: {limits.g_n_subsample} Ziehungen")
res_G = run_stage("G")
print({k: len(v) for k, v in res_G.items()})
manifest_G = substage_report("G")
cl = table("G", "clouds")
if len(cl):
    print("Realisierungen pro Bedingung (excluded = Konsistenzprüfung nicht bestanden):")
    show(cl.groupby(["group", "excluded"]).size().rename("n").reset_index(), n=60)


In [ ]:
# --- G0-Gate: Kern-und-Streuung-Ordnung exakter Übersetzungen (Kontrast 7) ---------------
# d(A_E, A_D) < d(A_E, B_E) < d(other family) under the truncated Mahalanobis distance of the
# prefix-mean states (deflation k=5); the gate passes when the order holds in >= 50 % of the
# snapshots (all / middle third).  `degenerate`: r_eff >= N-1, the distance is uninformative.
with guarded("G0-Gate"):
    gate = table("G", "g0_gate")
    show(gate)
    if len(gate) and truthy(gate["degenerate"]).iloc[0]:
        print("ACHTUNG: degenerate = True (r_eff ≥ N−1: zu wenige gepoolte Zustände für den Rang) -> "
              "Gate nicht aussagekräftig; mehr Familien/Formulierungen nötig (FAMILIES = None).")
    po = table("G", "pair_ordering")
    cats = [c for c in ["exact_translation", "same_meaning_same_lang", "other_family"] if c in set(po["category"])]
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
    for c in cats:
        g = po[po["category"] == c].sort_values("layer")
        a1.errorbar(g["layer"], g["mean_mahal"], yerr=g["sd_mahal"].fillna(0), marker="o", ms=4, capsize=2, label=c)
        a2.plot(g["layer"], g["mean_cosine"], marker="o", ms=4, label=c)
    for _, r in po[truthy(po["core_order_ok"])].drop_duplicates("layer").iterrows():
        a1.axvspan(r["layer"] - 0.4, r["layer"] + 0.4, color="green", alpha=0.08)
    a1.set_xlabel("Snapshot ℓ")
    a1.set_ylabel("mittlere Mahalanobis-Distanz (trunkiert, k=5)")
    a1.set_title("Kern (exakte Übersetzung) < Streuung (freie Realisierung) < Fremdfamilie?\n(grün: Ordnung hält)", fontsize=9)
    a1.legend(fontsize=7)
    a2.set_xlabel("Snapshot ℓ")
    a2.set_ylabel("mittlerer Kosinus")
    a2.set_title("Kosinus derselben Paare (Mean-Pool des Präfixes)", fontsize=9)
    a2.legend(fontsize=7)
    savefig("g0_pair_ordering")
    plt.show()


In [ ]:
# --- T1: Tiefen-Trajektorie der lokalen Geometrie (Log-Volumen, Inkremente, PR) ----------
with guarded("T1 Tiefen-Trajektorie"):
    dt = table("G", "depth_trajectory")
    k_def = 5 if 5 in set(dt["k"]) else int(dt["k"].iloc[0])
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    for (K, Cc), g in sorted(dt[(dt["readout"] == "mean") & (dt["k"] == k_def)].groupby(["K", "C"])):
        m = g.groupby("layer")[["log_volume", "delta_log_volume"]].mean()
        axes[0].plot(m.index, m["log_volume"], marker="o", ms=3, label=f"K={K} {Cc}")
        axes[1].plot(m.index, m["delta_log_volume"], marker="o", ms=3, label=f"K={K} {Cc}")
    for (K, Cc), g in sorted(dt[(dt["readout"] == "last") & (dt["k"] == k_def)].groupby(["K", "C"])):
        m = g.groupby("layer")["participation_ratio"].mean()
        axes[2].plot(m.index, m.values, marker="s", ms=3, label=f"K={K} {Cc}")
    axes[0].set_title(f"Log-Volumen der Realisierungswolke (Mean-Pool, k={k_def})", fontsize=9)
    axes[1].set_title("Inkrement Δ_ℓ log-Vol (negativ = Kontraktion; Vorzeichenwechsel = „Atmen“)", fontsize=9)
    axes[1].axhline(0, color="k", lw=0.6)
    axes[2].set_title(f"Participation Ratio (Last-Token, k={k_def})", fontsize=9)
    for ax in axes:
        ax.set_xlabel("Snapshot ℓ")
        ax.legend(fontsize=7)
    savefig("t1_depth_trajectory")
    plt.show()
    cols = [c for c in ["family_id", "K", "C", "readout", "k", "breathing_index", "breathing_amplitude",
                        "layer_of_max_contraction", "layer_of_max_expansion", "net_change"] if c in dt.columns]
    print("Atem-Index (Vorzeichenwechsel von Δ_ℓ log-Vol), Amplitude und Phasenlage pro Bedingung:")
    show(dt[dt["k"] == k_def].drop_duplicates(["family_id", "K", "C", "readout"])[cols], n=40)


In [ ]:
# --- G1: Kontraktion unter Sprachwechsel, n-angepasst, mit Familien-Bootstrap (Kontrast 8) --
# d_log_volume = log-Vol(K, C) - log-Vol(K=0) per family and snapshot; the mixed cloud is
# re-sampled to the K=0 cloud's size because the log-volume sums min(n, d) eigenvalues
# (raw values of clouds with different n are not comparable; both are reported).
with guarded("G1 Kontraktion"):
    cb = table("G", "contraction_boot")
    # confirmation set (test families F01-F12) when the run has it, else all families
    split = "test" if "split" in cb and (cb["split"] == "test").any() else "all"
    cb = cb[cb["split"] == split] if "split" in cb else cb
    k_def = 5 if 5 in set(cb["k"]) else int(cb["k"].iloc[0])
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for ax, (q, ro) in zip(axes, [("d_log_volume", "mean"), ("d_participation_ratio", "last")]):
        sub = cb[(cb["quantity"] == q) & (cb["readout"] == ro) & (cb["k"] == k_def)]
        for (K, Cc), g in sorted(sub.groupby(["K", "C"])):
            g = g.sort_values("layer")
            pre = bool(truthy(g["preregistered"]).any())
            ax.errorbar(g["layer"], g["mean"],
                        yerr=[(g["mean"] - g["ci_low"]).clip(lower=0).fillna(0), (g["ci_high"] - g["mean"]).clip(lower=0).fillna(0)],
                        marker="o", ms=4, capsize=2, lw=2 if pre else 1, label=f"K={K} {Cc}" + (" (präregistriert)" if pre else ""))
        ax.axhline(0, color="k", lw=0.6)
        ax.set_xlabel("Snapshot ℓ")
        ax.set_title(f"{q} ({ro}-Readout, k={k_def}, Familien: {split}); 95 %-CI aus {limits.n_boot} "
                     f"Familien-Bootstraps", fontsize=9)
        ax.legend(fontsize=7)
    savefig("g1_contraction")
    plt.show()
    con = table("G", "contraction")
    pre = con[truthy(con["preregistered"])]
    if len(pre):
        print("Roh vs. n-angepasst (präregistrierte Zeilen K=8/C1/k=5, Mittel über Familien):")
        show(pre.groupby(["readout", "layer"])[["n_mixed", "n_ref", "d_log_volume_raw", "d_log_volume",
                                                  "d_participation_ratio_raw", "d_participation_ratio"]].mean())


In [ ]:
# --- T2: Positions-Trajektorie — Log-Volumen pro Position (Präfix-Bins, ab Wechselstelle, Tail) --
with guarded("T2 Positions-Trajektorie"):
    pt = table("G", "position_traj")
    site0 = C.dossier_site(mb).key if C.dossier_site(mb).key in set(pt["site"]) else str(pt["site"].iloc[0])
    sub = pt[pt["site"] == site0]
    regions = [r for r in ["prefix_bin", "from_switch", "tail"] if r in set(sub["region"])]
    titles = {"prefix_bin": "Präfix (relative Positionsbins)", "from_switch": "Position relativ zur letzten Wechselstelle",
              "tail": "gemeinsamer Schlussabschnitt (vom Ende her)"}
    fig, axes = plt.subplots(1, len(regions), figsize=(5 * len(regions), 3.8), squeeze=False)
    for ax, region in zip(axes[0], regions):
        for grp, g in sorted(sub[sub["region"] == region].groupby("group")):
            g = g.sort_values("index")
            ax.plot(g["index"], g["log_volume"], marker="o", ms=3, label=grp)
        ax.set_title(titles[region], fontsize=9)
        ax.set_xlabel("Index")
        ax.set_ylabel("Log-Volumen (Last-Token je Position)")
        ax.legend(fontsize=6)
    fig.suptitle(f"T2 an {site0}: wandert die Verdichtung mit den Wechselstellen?", fontsize=10)
    savefig("t2_position_trajectory")
    plt.show()


In [ ]:
# --- T3: Steering-Validität — cos(Steuerverschiebung, mean(K=8) − mean(K=0)) (Kontrast 9) --
with guarded("T3 Steering"):
    st = table("G", "steering")
    sb = table("G", "steering_boot")
    eps0 = 0.05 if np.isclose(st["eps"], 0.05).any() else float(st["eps"].iloc[0])
    sites = list(dict.fromkeys(st["inject_site"]))
    fig, axes = plt.subplots(1, len(sites), figsize=(6 * len(sites), 4), squeeze=False, sharey=True)
    for ax, isk in zip(axes[0], sites):
        g = st[(st["inject_site"] == isk) & np.isclose(st["eps"], eps0)]
        m = g.groupby("layer")[["cos_steer_vs_K", "cos_rand_mean", "cos_rand_sd"]].mean()
        ax.plot(m.index, m["cos_steer_vs_K"], "o-", label="lang_dom (Mittel über Familien)")
        ax.fill_between(m.index, m["cos_rand_mean"] - 2 * m["cos_rand_sd"], m["cos_rand_mean"] + 2 * m["cos_rand_sd"],
                        color="grey", alpha=0.25, label="Zufallsrichtungen ± 2 SD")
        ax.axhline(0.3, color="r", lw=0.8, ls="--", label="Mindesteffekt cos ≥ 0,3")
        ax.axhline(0, color="k", lw=0.6)
        for b in sorted(set(g[truthy(g["at_max_disp_layer"])]["layer"])):
            ax.axvline(b, color="C1", lw=0.8, ls=":")
        ax.set_title(f"Injektion {isk}, ε={eps0:g}", fontsize=9)
        ax.set_xlabel("Snapshot ℓ")
        ax.legend(fontsize=7)
    axes[0, 0].set_ylabel("cos(Steuerverschiebung, mean(K=8) − mean(K=0))")
    fig.suptitle("T3: reproduziert kontinuierliches Steuern die Geometrie echter Wechsel? "
                 "(gepunktet: Layer maximaler diskreter Verschiebung)", fontsize=10)
    savefig("t3_steering_cosine")
    plt.show()
    top = sb[sb["quantity"] == "cos_at_max_disp_layer"]
    if len(top):
        print("Präregistrierter Kontrast 9: Kosinus im Layer maximaler Verschiebung (Familien-Bootstrap):")
        show(top[[c for c in ["inject_site", "eps", "split", "preregistered", "confirmatory", "min_effect_met", "mean",
                              "ci_low", "ci_high", "n_clusters"] if c in top.columns]])


In [ ]:
# --- T5: Kopplung von Δ log-Vol mit λ_ℓ (D1) und Lens-KL (D2) über die Layer --------------
with guarded("T5 Kopplung"):
    cp = table("G", "coupling")
    if cp.empty:
        print("Keine Kopplung: Stufe D ist für dieses Modell unter RUNS_ROOT noch nicht gelaufen (Notebook 03) "
              "oder die Tabellen damage_profile / lens fehlen.")
    else:
        show(cp.sort_values(["partner", "K", "C"]), n=60)
        fig, ax = plt.subplots(figsize=(7, 3.6))
        for partner, g in cp.groupby("partner"):
            lab = {"lambda": "λ_ℓ (Störungswachstum, Stufe D)", "kl": "Lens-KL (Stufe D)"}.get(partner, partner)
            ax.scatter([f"K={k} {c}" for k, c in zip(g["K"], g["C"])], g["rho"], label=lab, s=30)
        ax.axhline(0, color="k", lw=0.6)
        ax.set_ylabel("Spearman ρ über Layer")
        ax.set_title("Ist Verdichtung der Ordnungsparameter der Nicht-Gleichgewichts-Signaturen?", fontsize=9)
        ax.legend(fontsize=7)
        savefig("t5_coupling")
        plt.show()


In [ ]:
# --- Kurze Laufzusammenfassung nach OUT_ROOT schreiben ---------------------------------
with guarded("Zusammenfassung schreiben"):
    import datetime

    # failed substages (stages G-L) from the manifests: a stage whose table is an empty default
    # must not read plain "ok" in the summary (reused stages included)
    for st, r in report.items():
        if r.get("status") in ("ok", "reused") and r.get("run_dir"):
            r.update(C.substage_status(r["run_dir"], r["status"]))
    substage_errors = {st: r["substage_errors"] for st, r in report.items() if r.get("substage_errors")}
    status_overall = ("error" if any(r.get("status") == "error" for r in report.values())
                      else "substage_errors" if substage_errors else "ok")
    if substage_errors:
        print("ACHTUNG, fehlgeschlagene Substufen:", substage_errors)
    summary = dict(
        notebook=NOTEBOOK, stage=STAGE, model=spec.name, revision=spec.revision, dtype=spec.dtype,
        device=str(mb.device), n_layers=mb.n_layers, d_model=mb.d_model, smoke=bool(SMOKE),
        families=FAMILIES, seed=SEED, out_root=OUT_ROOT, runs_root=RUNS_ROOT, cli_command=CLI_COMMAND,
        timestamp=datetime.datetime.now(datetime.timezone.utc).isoformat(),
        status_overall=status_overall, substage_errors=substage_errors,
        report=report,
        tables={st: {n: int(len(df)) for n, df in r.items()} for st, r in results.items()},
        figures=sorted(f for f in os.listdir(FIG_DIR) if f.endswith(".png")),
    )
    path = os.path.join(OUT_ROOT, f"summary_{NOTEBOOK}_{spec.tag().replace(':', '_').replace('@', '_')}.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(C.to_jsonable(summary), f, indent=1, ensure_ascii=False)
    print("Zusammenfassung:", path)
    print(json.dumps(C.to_jsonable(summary), indent=1, ensure_ascii=False)[:3000])
